# Ecoregion intactness

Data on continuous intactness can be retrieved from [UQ eSpace](https://espace.library.uq.edu.au/view/UQ:f51cace) and presents three layers:

- Ecoregion intactness scores (shapefile) - stored in data/raw as Ecoregions2017_intactness
- Intactness (raster) - stored as GEE asset in projects/gmvad-grass/assets/intactness/intactness_Qprime_2009
- Changes in intactness (raster) - stored as GEE asset in projects/gmvad-grass/assets/intactness/intactness_Qslope_2009

Dataset reference:[Beyer, H. L., Venter, O., Grantham, H. S., Watson, J. E. M. (2020) Substantial losses in ecoregion intactness highlights urgency of globally coordinated action. Conservation Letters 13:e12692](https://doi.org/10.1111/conl.12692])

The **purpose** of this notebook is to process the shapefile (and convert it to MBTiles) so that includes the necessary fields to display these 3 maps:

- Global distribution of intactness scores for terrestrial ecoregions in 2009 (Fig 2a in paper)
- Absolute change in intactness between 1993 and 2009 (Fig 2b in paper)
- Map with trend in their intactness between 1993 and 2009 (degrading, stable, or improving) and the proportion of habitat remaining in the ecoregion that had an HFI value of less than four (poor: <33%, moderate: 33–67%. high: >67%) (Fig 3b).

The fields used for generating the first two maps are *PRHAB1993* and *PRHAB2009*, which estimate of the proportion of habitat in 1993 and 2009 using a binary interpretation of habitat. They have to be **multiplied by 100** to get the numbers displayed on the paper's figures.

For the third map, the *PLOTCAT* already provides the required categories, so the styling can be done directly in Strapi using this column.

### Set up
#### Library import

In [ ]:
import os
import geopandas as gpd
import pandas as pd
from pathlib import Path
import logging
import subprocess
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors

# Create a logger
logger = logging.getLogger(__name__)

# Set the log level to INFO
logger.setLevel(logging.INFO)

#### Utils

In [ ]:
# Function to create MBTILES from a GeoJSON file
def create_mbtiles(
    source_path: Path,
    output_path: Path,
    layer_name: str,
    max_zoom: int,
    opts="--read-parallel --no-tile-compression -s EPSG:4326 -B4",
):
    """
    Use tippecanoe to create pbf tiles at dest_path from source_path (geojson).
    layer_name is used for the name of the layer in the MBTILE.
    Regex file path (/*.geojson) is supported for source_path.
    This function replaces the previous two functions (create_mbtiles & mbtile_to_pbf).

    More info: https://github.com/mapbox/tippecanoe#options

    Args:
        source_path (Path): path to source geojson
        output_path (Path): path to output .mbtiles
        layer_name (str): name of layer in the MBTILE
        max_zoom (int): max zoom level
        opts (str): options for tippecanoe

    Returns:
        (int): 0 if the file was created successfully, 1 if the file creation failed.
    """
    try:
        opts += f" -z{max_zoom}"
        cmd = f"tippecanoe -o {output_path} -l {layer_name} {opts} {source_path}"
        logger.info(f"Processing: {cmd}")
        r = subprocess.call(cmd, shell=True)
        if r == 0:
            logger.info("Task created")
        return r

    except Exception as e:
        logger.error(e)
        return 1

### Processing
#### Load and prepare data

In [ ]:
intact = gpd.read_file("../data/raw/Ecoregions2017_intactness/Ecoregions2017_intactness.shp")
intact = intact.to_crs(epsg=4326)
intact.plot()

In [ ]:
# Convert columns Q1993 and Q2009 to numeric and multiply by 100
intact["PRHAB1993"] = pd.to_numeric(intact["PRHAB1993"], errors="coerce") * 100 
intact["PRHAB2009"] = pd.to_numeric(intact["PRHAB2009"], errors="coerce") * 100

In [ ]:
# Define the bins and labels
bins = [0, 10, 20, 30, 40, 50, 60, 70, 80, 90, 100]
labels = [
    '0 - 10', '10 - 20', '20 - 30', '30 - 40', '40 - 50',
    '50 - 60', '60 - 70', '70 - 80', '80 - 90', '90 - 100'
]
colors = [
    '#E81015', '#F35625', '#FA8633', '#FCB344', '#FCE55B',
    '#E7ED73', '#BFD38B', '#95BEA0', '#68A6B3', '#2892C6'
]

# Bin the PRHAB2009 values
intact['PRHAB2009_bins'] = pd.cut(intact['PRHAB2009'], bins=bins, labels=labels, include_lowest=True)

# Plot the map to check it has the same colors as Fig 2a
fig, ax = plt.subplots(1, 1, figsize=(15, 10))
intact.plot(ax=ax, column='PRHAB2009_bins', cmap=mcolors.ListedColormap(colors), legend=True, legend_kwds={'bbox_to_anchor': (1, 0), 'loc': 'lower left', 'title': 'Intactness 2009'})
plt.title('Intactness 2009')
plt.show()

#### Calculate change 2009 - 1993

In [ ]:
# Calculate change between 2009 and 1993
intact["PRHAB_change"] = intact["PRHAB2009"] - intact["PRHAB1993"]

In [ ]:
# Define the bins and labels for PRHAB_change
change_bins = [-float('inf'), -30, -15, -5, -1, 1, 5, 15, 30, float('inf')]
change_labels = [
    '< -30%', '-30 - -15%', '-15 - -5%', '-5 - -1%', '-1 - 1%',
    '1 - 5%', '5 - 15%', '15 - 30%', '> 30%'
]
change_colors = [
    '#E81015', '#FF6600', '#FE9900', '#FFCC00', '#E1E1E1',
    '#B7D400', '#7AAB01', '#3F8200', '#006100'
]

# Bin the PRHAB_change values
intact['PRHAB_change_bins'] = pd.cut(intact['PRHAB_change'], bins=change_bins, labels=change_labels, include_lowest=True)

# Plot the map for PRHAB_change to check it has the colors in Fig 2b
fig, ax = plt.subplots(1, 1, figsize=(15, 10))
intact.plot(ax=ax, column='PRHAB_change_bins', cmap=mcolors.ListedColormap(change_colors), legend=True, legend_kwds={'bbox_to_anchor': (1, 0), 'loc': 'lower left', 'title': 'Intactness 2009 - 1993'})
plt.title('Intactness 2009 - 1993')
plt.show()

In [ ]:
# Convert categorical columns to strings
intact['PRHAB2009_bins'] = intact['PRHAB2009_bins'].astype(str)
intact['PRHAB_change_bins'] = intact['PRHAB_change_bins'].astype(str)

# Save as geojson 
intact.to_file("../data/processed/intactness.geojson", driver="GeoJSON")

### Create mbtiles

In [ ]:
path = '../data/processed'

create_mbtiles(
    os.path.join(path, "intactness.geojson"),
    os.path.join(path, "intactness.mbtiles"),
    "intactness",
    12,
    "--force --read-parallel -zg -Z2 --drop-densest-as-needed --extend-zooms-if-still-dropping",
)
